In [62]:
%pip install trino pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: C:\Users\Михаил\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [63]:
from trino.dbapi import connect
import pandas as pd

conn = connect(
    host="153.51.166.221",
    port=8080,
    user="sergeev_me",
    catalog="lakehouse",
    schema="raw",
    http_scheme="http"
)

print("Подключение успешно!")

Подключение успешно!


In [64]:
cursor = conn.cursor()

cursor.execute("SHOW TABLES FROM lakehouse.raw")

tables = cursor.fetchall()

df_tables = pd.DataFrame(tables, columns=["table_name"])

display(df_tables)

,table_name
0,dota_duration_15min_gold_events
1,dota_duration_15min_kill_events
2,dota_duration_15min_match_players
3,dota_duration_15min_matches
4,dota_duration_15min_tower_damage_events
5,dota_duration_15min_tower_death_events
6,mongo__ability_learn_events
7,mongo__ability_used_events
8,mongo__assist_events
9,mongo__building_events



## Методика и метрики

Это описательный статистический анализ, а не причинная модель.

Для каждого матча определяется время первого уничтожения башни T1 противника. Затем команда, уничтожившая башню, сравнивается с командой-победителем матча.

Основная метрика — `first_tower_win_rate`: доля матчей, в которых команда, уничтожившая первую башню T1, в итоге выиграла.

Дополнительно используются количество матчей в группе и среднее время уничтожения первой башни.

Матчи группируются по временным интервалам:
- до 5 минут;
- 5–10 минут;
- 10–15 минут;
- 15–20 минут;
- после 20 минут.

Результаты показывают статистическую связь между временем получения преимущества и итогом матча, но не доказывают причинность.



## 1. Источники и детализация данных

Используются таблицы Trino Lakehouse:

- PostgreSQL `pg__matches` — информация о матчах, продолжительность и победившая команда.
- MongoDB `mongo__tower_death_events` — события изменения состояния построек.
- PostgreSQL `pg__npcs` — справочник игровых объектов для идентификации башен T1.

Дополнительно может использоваться MongoDB `mongo__tower_damage_events` для проверки корректности определения башен.

Основной ключ между таблицами — `match_id`.

Детализация будущей STAGE-таблицы — одна запись на матч.



### 1.1. Проверка исходных таблиц

Проверяем количество записей, уникальных матчей, диапазоны времени и наличие пропусков в ключевых полях.

Для справочника NPC отдельно проверяем наличие башен первого уровня (T1).


In [65]:

# Основные таблицы нашего кейса
source_tables = {
    "PostgreSQL — матчи": "lakehouse.raw.pg__matches",
    "MongoDB — события уничтожения башен": "lakehouse.raw.mongo__tower_death_events",
    "PostgreSQL — справочник NPC": "lakehouse.raw.pg__npcs"
}

# Дополнительная таблица для проверки
optional_tables = {
    "MongoDB — события урона башням": "lakehouse.raw.mongo__tower_damage_events"
}

# Проверяем, какие таблицы существуют в RAW
cursor.execute("SHOW TABLES FROM lakehouse.raw")
available_tables = {row[0] for row in cursor.fetchall()}


def inspect_table(description, table_name):

    print("\n" + "=" * 90)
    print(description)
    print(f"Таблица: {table_name}")
    print("=" * 90)

    # 1. Получаем структуру таблицы
    cursor.execute(f"DESCRIBE {table_name}")

    schema_columns = [column[0] for column in cursor.description]
    schema_data = cursor.fetchall()

    print("\nСТРУКТУРА ТАБЛИЦЫ:")
    display(pd.DataFrame(schema_data, columns=schema_columns))

    # 2. Получаем первые 10 записей
    cursor.execute(f"SELECT * FROM {table_name} LIMIT 10")

    data_columns = [column[0] for column in cursor.description]
    data = cursor.fetchall()

    print("\nПРИМЕРЫ ДАННЫХ (первые 10 записей):")
    display(pd.DataFrame(data, columns=data_columns))


# Вывод основных таблиц
print("ОСНОВНЫЕ ИСТОЧНИКИ ДАННЫХ")

for description, table_name in source_tables.items():

    short_name = table_name.split(".")[-1]

    if short_name in available_tables:
        inspect_table(description, table_name)
    else:
        print(f"\nТаблица не найдена: {table_name}")


# Вывод дополнительной таблицы, если она существует
print("\nДОПОЛНИТЕЛЬНЫЕ ИСТОЧНИКИ ДАННЫХ")

for description, table_name in optional_tables.items():

    short_name = table_name.split(".")[-1]

    if short_name in available_tables:
        inspect_table(description, table_name)
    else:
        print(f"\nДополнительная таблица отсутствует: {table_name}")


ОСНОВНЫЕ ИСТОЧНИКИ ДАННЫХ

PostgreSQL — матчи
Таблица: lakehouse.raw.pg__matches

СТРУКТУРА ТАБЛИЦЫ:


,Column,Type,Extra,Comment
0,barracks_status_dire,integer,,Битовая маска казарм Dire
1,barracks_status_radiant,integer,,Битовая маска казарм Radiant
2,cluster,integer,,ID кластера (регион)
3,dire_score,integer,,Убийств Dire
4,discover_run_id,integer,,ID discover-запуска
5,duration,integer,,Длительность (сек)
6,enrich_run_id,integer,,ID enrich-запуска
7,enrichment_attempts,integer,,Попыток обогащения (макс. 5)
8,first_blood_time,integer,,Время первого убийства (сек)
9,game_mode,integer,,ID режима игры



ПРИМЕРЫ ДАННЫХ (первые 10 записей):


,barracks_status_dire,barracks_status_radiant,cluster,dire_score,discover_run_id,duration,enrich_run_id,enrichment_attempts,first_blood_time,game_mode,...,match_id,match_seq_num,radiant_score,radiant_win,start_time,tower_status_dire,tower_status_radiant,job_id,sys_insert_stamp,src_modified_stamp
0,63,51,153,52,1816,2284,1672,0,189,22,...,9045678812,7597144944,14,False,1792043270,1983,260,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 19:20:56.749827+00:00
1,0,63,251,21,1816,2268,1672,0,160,22,...,9045678910,7597145052,44,True,1792043269,0,1974,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 19:21:06.158468+00:00
2,0,63,154,21,1815,2135,1672,0,147,22,...,9045679000,7597144165,55,True,1792043264,0,1983,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 19:21:15.425029+00:00
3,0,63,151,28,1814,2529,1670,0,107,22,...,9045674548,7597143676,47,True,1792042819,0,1975,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 18:36:49.200541+00:00
4,16,63,251,45,1814,2698,1670,0,128,22,...,9045673072,7597143900,50,True,1792042660,0,1972,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 18:38:06.998130+00:00
5,63,0,171,55,1814,1182,1670,0,7,23,...,9045688727,7597144073,24,False,1792044309,2039,0,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 18:20:27.780885+00:00
6,3,63,152,15,1814,1204,1670,0,173,23,...,9045688572,7597144086,38,True,1792044305,4,1982,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 18:20:30.473355+00:00
7,48,63,144,6,1814,1105,1670,0,83,23,...,9045688720,7597143666,57,True,1792044294,256,2039,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 18:20:32.300943+00:00
8,63,48,144,34,1814,1270,1670,0,108,23,...,9045687832,7597144036,14,False,1792044225,2038,0,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 18:20:44.198495+00:00
9,48,63,182,10,1814,1249,1670,0,53,23,...,9045687562,7597143671,45,True,1792044199,256,2038,789fe435-1f9b-48e0-87e3-accb183cb37b,2026-11-13 05:42:40.830110+00:00,2026-11-12 18:20:55.239309+00:00



MongoDB — события уничтожения башен
Таблица: lakehouse.raw.mongo__tower_death_events

СТРУКТУРА ТАБЛИЦЫ:


,Column,Type,Extra,Comment
0,dire,integer,,Битовая маска башен Dire
1,id,varchar,,Уникальный ID документа
2,match_id,bigint,,ID матча
3,radiant,integer,,Битовая маска башен Radiant
4,time,integer,,Время уничтожения (сек)
5,job_id,varchar,,Идентификатор прогона загрузки (UUID)
6,sys_insert_stamp,timestamp(6) with time zone,,Время физической загрузки строки в озеро
7,src_modified_stamp,timestamp(6) with time zone,,Метка времени изменения в источнике


... (6 outputs left out)


ДОПОЛНИТЕЛЬНЫЕ ИСТОЧНИКИ ДАННЫХ

MongoDB — события урона башням
Таблица: lakehouse.raw.mongo__tower_damage_events

СТРУКТУРА ТАБЛИЦЫ:


,Column,Type,Extra,Comment
0,attacker,integer,,steamAccountId атакующего
1,byability,integer,,ID способности
2,byitem,integer,,ID предмета
3,damage,integer,,Количество урона
4,fromnpc,integer,,ID NPC-источника
5,id,varchar,,Уникальный ID документа
6,match_id,bigint,,ID матча
7,npcid,integer,,ID башни
8,steam_account_id,bigint,,Steam account ID
9,time,integer,,Время урона (сек)



ПРИМЕРЫ ДАННЫХ (первые 10 записей):


,attacker,byability,byitem,damage,fromnpc,id,match_id,npcid,steam_account_id,time,job_id,sys_insert_stamp,src_modified_stamp
0,75,0,0,28,0,6ad5517d5ce156150e08ef04,9044689256,29,301865375,1183,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
1,75,0,0,27,0,6ad5517d5ce156150e08ef05,9044689256,29,301865375,1184,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
2,75,0,0,30,0,6ad5517d5ce156150e08ef07,9044689256,37,301865375,1776,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
3,75,0,0,26,0,6ad5517d5ce156150e08ef09,9044689256,33,301865375,1810,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
4,75,0,0,17,0,6ad5517d5ce156150e08ef0b,9044689256,33,301865375,1829,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
5,75,0,0,27,0,6ad5517d5ce156150e08ef0f,9044689256,45,301865375,1837,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
6,75,0,0,31,0,6ad5517d5ce156150e08ef13,9044689256,45,301865375,1840,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
7,75,0,0,34,0,6ad5517d5ce156150e08ef14,9044689256,45,301865375,1841,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
8,75,0,0,34,0,6ad5517d5ce156150e08ef15,9044689256,45,301865375,1842,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00
9,75,0,0,35,0,6ad5517d5ce156150e08ef17,9044689256,45,301865375,1844,53e30e02-34a8-4899-a0c7-ab32942d6462,2026-10-19 00:18:44.795731+00:00,2026-10-18 23:08:45.209000+00:00


In [66]:

# Аудит качества RAW

quality_checks = {

    "1. Качество матчей": """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT match_id) AS unique_match_ids,
            COUNT_IF(match_id IS NULL) AS null_match_ids,
            COUNT_IF(duration IS NULL OR duration <= 0)
                AS invalid_duration,
            COUNT_IF(radiant_win IS NULL)
                AS null_winners
        FROM lakehouse.raw.pg__matches
    """,

    "2. Качество событий уничтожения башен": """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT id) AS unique_event_ids,
            COUNT_IF(id IS NULL) AS null_event_ids,
            COUNT_IF(match_id IS NULL) AS null_match_ids,
            COUNT_IF(time IS NULL) AS null_times,
            COUNT_IF(time < 0) AS negative_times,
            COUNT_IF(dire IS NULL) AS null_dire_masks,
            COUNT_IF(radiant IS NULL) AS null_radiant_masks,
            COUNT_IF(id IS NOT NULL) - COUNT(DISTINCT id)
                AS duplicate_event_ids
        FROM lakehouse.raw.mongo__tower_death_events
    """,

    "3. Проверка связей и времени событий": """
        SELECT
            COUNT(*) AS total_events,

            COUNT_IF(m.match_id IS NULL)
                AS events_without_match,

            COUNT_IF(
                m.match_id IS NOT NULL
                AND d.time > m.duration
            ) AS events_after_match_end,

            COUNT_IF(d.time < 0)
                AS events_with_negative_time

        FROM lakehouse.raw.mongo__tower_death_events d

        LEFT JOIN lakehouse.raw.pg__matches m
            ON d.match_id = m.match_id
    """,

    "4. Проверка последовательности масок": """
        WITH ordered AS (
            SELECT
                match_id,
                time,
                id,
                dire,
                radiant,

                LAG(dire) OVER (
                    PARTITION BY match_id
                    ORDER BY time, id
                ) AS prev_dire,

                LAG(radiant) OVER (
                    PARTITION BY match_id
                    ORDER BY time, id
                ) AS prev_radiant

            FROM lakehouse.raw.mongo__tower_death_events
        )

        SELECT
            COUNT(*) AS total_events,

            COUNT_IF(
                dire IS NOT NULL
                AND prev_dire IS NOT NULL
                AND bitwise_and(dire, prev_dire) <> prev_dire
            ) AS dire_mask_regressions,

            COUNT_IF(
                radiant IS NOT NULL
                AND prev_radiant IS NOT NULL
                AND bitwise_and(radiant, prev_radiant) <> prev_radiant
            ) AS radiant_mask_regressions

        FROM ordered
    """
}


for check_name, query in quality_checks.items():

    print("\n" + "=" * 75)
    print(check_name)
    print("=" * 75)

    cursor.execute(query)

    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()

    display(pd.DataFrame(rows, columns=columns))



1. Качество матчей


,total_rows,unique_match_ids,null_match_ids,invalid_duration,null_winners
0,317104,317104,0,0,0



2. Качество событий уничтожения башен


,total_rows,unique_event_ids,null_event_ids,null_match_ids,null_times,negative_times,null_dire_masks,null_radiant_masks,duplicate_event_ids
0,7692290,7692290,0,0,0,0,0,0,0



3. Проверка связей и времени событий


,total_events,events_without_match,events_after_match_end,events_with_negative_time
0,7692290,35557,0,0



4. Проверка последовательности масок


,total_events,dire_mask_regressions,radiant_mask_regressions
0,7692290,0,0


In [67]:

query = """
WITH event_counts AS (
    SELECT
        match_id,
        COUNT(*) AS event_count
    FROM lakehouse.raw.mongo__tower_death_events
    GROUP BY match_id
),

match_data AS (
    SELECT
        m.match_id,
        m.duration,
        COALESCE(e.event_count, 0) AS event_count,

        CASE
            WHEN m.duration < 300 THEN 'До 5 минут'
            WHEN m.duration < 600 THEN '5-10 минут'
            WHEN m.duration < 900 THEN '10-15 минут'
            WHEN m.duration < 1800 THEN '15-30 минут'
            WHEN m.duration < 2700 THEN '30-45 минут'
            ELSE '45+ минут'
        END AS duration_group,

        CASE
            WHEN m.duration < 300 THEN 1
            WHEN m.duration < 600 THEN 2
            WHEN m.duration < 900 THEN 3
            WHEN m.duration < 1800 THEN 4
            WHEN m.duration < 2700 THEN 5
            ELSE 6
        END AS group_order

    FROM lakehouse.raw.pg__matches m
    LEFT JOIN event_counts e
        ON m.match_id = e.match_id
)

SELECT
    duration_group,
    COUNT(*) AS total_matches,

    COUNT_IF(event_count > 0) AS matches_with_events,

    COUNT_IF(event_count = 0) AS matches_without_events,

    ROUND(
        100.0 * COUNT_IF(event_count = 0) / COUNT(*),
        2
    ) AS without_events_pct

FROM match_data
GROUP BY duration_group, group_order
ORDER BY group_order
"""

cursor.execute(query)

columns = [c[0] for c in cursor.description]
df_duration = pd.DataFrame(cursor.fetchall(), columns=columns)

display(df_duration)


# Отдельно смотрим короткие матчи
query_short = """
WITH event_counts AS (
    SELECT
        match_id,
        COUNT(*) AS event_count
    FROM lakehouse.raw.mongo__tower_death_events
    GROUP BY match_id
)

SELECT
    m.match_id,
    m.duration,
    m.radiant_win,
    COALESCE(e.event_count, 0) AS event_count

FROM lakehouse.raw.pg__matches m

LEFT JOIN event_counts e
    ON m.match_id = e.match_id

WHERE m.duration < 900

ORDER BY m.duration

LIMIT 30
"""

cursor.execute(query_short)

columns = [c[0] for c in cursor.description]
df_short = pd.DataFrame(cursor.fetchall(), columns=columns)

print("ПРИМЕРЫ МАТЧЕЙ КОРОЧЕ 15 МИНУТ:")
display(df_short)


,duration_group,total_matches,matches_with_events,matches_without_events,without_events_pct
0,5-10 минут,358,358,0,0.0
1,10-15 минут,1793,1793,0,0.0
2,15-30 минут,101467,101467,0,0.0
3,30-45 минут,151157,150945,212,0.1
4,45+ минут,62329,61719,610,1.0


ПРИМЕРЫ МАТЧЕЙ КОРОЧЕ 15 МИНУТ:


,match_id,duration,radiant_win,event_count
0,9045507701,374,False,2
1,9045267055,374,True,6
2,9044788300,374,True,3
3,9044836725,376,False,1
4,9044751733,376,False,3
5,9044806258,377,False,3
6,9045020640,382,False,1
7,9044887129,382,True,3
8,9044855329,385,False,3
9,9044975904,385,False,1


In [68]:

query = """
WITH ranked_events AS (
    SELECT
        match_id,
        dire,
        radiant,
        ROW_NUMBER() OVER (
            PARTITION BY match_id
            ORDER BY time DESC, id DESC
        ) AS rn
    FROM lakehouse.raw.mongo__tower_death_events
),

last_events AS (
    SELECT *
    FROM ranked_events
    WHERE rn = 1
)

SELECT
    COUNT(*) AS matched_matches,

    COUNT_IF(
        bitwise_and(bitwise_not(e.dire), 2047)
        = m.tower_status_dire
    ) AS dire_same_side_matches,

    COUNT_IF(
        bitwise_and(bitwise_not(e.dire), 2047)
        = m.tower_status_radiant
    ) AS dire_opposite_side_matches,

    COUNT_IF(
        bitwise_and(bitwise_not(e.radiant), 2047)
        = m.tower_status_radiant
    ) AS radiant_same_side_matches,

    COUNT_IF(
        bitwise_and(bitwise_not(e.radiant), 2047)
        = m.tower_status_dire
    ) AS radiant_opposite_side_matches

FROM last_events e
JOIN lakehouse.raw.pg__matches m
    ON e.match_id = m.match_id
"""

cursor.execute(query)

columns = [c[0] for c in cursor.description]
df_mask_mapping = pd.DataFrame(
    cursor.fetchall(),
    columns=columns
)

display(df_mask_mapping)



,matched_matches,dire_same_side_matches,dire_opposite_side_matches,radiant_same_side_matches,radiant_opposite_side_matches
0,316282,117120,594,122065,558


In [69]:

query = """
SELECT
    CASE
        WHEN duration < 900 THEN 'До 15 минут'
        ELSE '15 минут и более'
    END AS duration_group,

    lobby_type,
    game_mode,

    COUNT(*) AS matches_count,
    MIN(duration) AS min_duration,
    MAX(duration) AS max_duration

FROM lakehouse.raw.pg__matches

GROUP BY
    CASE
        WHEN duration < 900 THEN 'До 15 минут'
        ELSE '15 минут и более'
    END,
    lobby_type,
    game_mode

ORDER BY
    duration_group,
    lobby_type,
    game_mode
"""

cursor.execute(query)

columns = [c[0] for c in cursor.description]
df_modes = pd.DataFrame(cursor.fetchall(), columns=columns)

display(df_modes)


# Отдельная сводка по коротким матчам
query_short = """
SELECT
    COUNT(*) AS total_short_matches,

    COUNT_IF(lobby_type = 7)
        AS ranked_short_matches,

    COUNT_IF(game_mode = 23)
        AS turbo_short_matches,

    COUNT_IF(
        lobby_type = 7 AND game_mode <> 23
    ) AS ranked_non_turbo_short_matches

FROM lakehouse.raw.pg__matches
WHERE duration < 900
"""

cursor.execute(query_short)

columns = [c[0] for c in cursor.description]
display(pd.DataFrame(cursor.fetchall(), columns=columns))


,duration_group,lobby_type,game_mode,matches_count,min_duration,max_duration
0,15 минут и более,0,3,506,989,4114
1,15 минут и более,0,4,11624,900,5383
2,15 минут и более,0,22,27883,901,5759
3,15 минут и более,0,23,100280,900,4679
4,15 минут и более,1,1,1,2144,2144
5,15 минут и более,1,2,193,990,4498
6,15 минут и более,1,3,4,1732,3382
7,15 минут и более,1,4,2,2559,2642
8,15 минут и более,1,16,40,945,4786
9,15 минут и более,1,22,12,960,3109


,total_short_matches,ranked_short_matches,turbo_short_matches,ranked_non_turbo_short_matches
0,2151,159,1806,159


In [70]:

query = """
WITH ranked AS (
    SELECT
        d.match_id,
        d.id,
        d.dire,
        d.radiant,

        ROW_NUMBER() OVER (
            PARTITION BY d.match_id
            ORDER BY d.time DESC, d.id DESC
        ) AS rn

    FROM lakehouse.raw.mongo__tower_death_events d
),

last_events AS (
    SELECT *
    FROM ranked
    WHERE rn = 1
)

SELECT
    COUNT(*) AS matched_matches,

    COUNT_IF(
        bitwise_and(bitwise_not(dire), 2047)
        = m.tower_status_dire
    ) AS dire_11_bits,

    COUNT_IF(
        bitwise_and(bitwise_not(dire), 65535)
        = m.tower_status_dire
    ) AS dire_16_bits,

    COUNT_IF(
        bitwise_and(bitwise_not(dire), 131071)
        = m.tower_status_dire
    ) AS dire_17_bits,

    COUNT_IF(
        bitwise_and(bitwise_not(radiant), 2047)
        = m.tower_status_radiant
    ) AS radiant_11_bits,

    COUNT_IF(
        bitwise_and(bitwise_not(radiant), 65535)
        = m.tower_status_radiant
    ) AS radiant_16_bits,

    COUNT_IF(
        bitwise_and(bitwise_not(radiant), 131071)
        = m.tower_status_radiant
    ) AS radiant_17_bits

FROM last_events e

JOIN lakehouse.raw.pg__matches m
    ON e.match_id = m.match_id
"""

cursor.execute(query)

columns = [c[0] for c in cursor.description]
display(pd.DataFrame(cursor.fetchall(), columns=columns))


,matched_matches,dire_11_bits,dire_16_bits,dire_17_bits,radiant_11_bits,radiant_16_bits,radiant_17_bits
0,316282,117120,83166,82323,122065,84530,83438


Что будет делать STAGE
1. Берёт все матчи из pg__matches.
2. Присоединяет события из mongo__tower_death_events по match_id.
3. Учитывает только события с корректным временем в пределах матча.
4. Устраняет дубликаты событий по id.
5. Находит первое изменение младших трёх битов масок — наша рабочая логика определения падения T1.
6. Определяет уничтоженную сторону и противоположную команду-уничтожителя.
7. Сохраняет одну строку на матч, не включая матчи без событий T1.
8. НЕ учитываем матчи в которых башни упали в одну секунду

In [71]:

from trino.dbapi import connect
import uuid

# Создаём новое соединение с увеличенным таймаутом
try:
    conn.close()
except Exception:
    pass

conn = connect(
    host="153.51.166.221",
    port=8080,
    user="sergeev_me",
    catalog="lakehouse",
    schema="stage",
    http_scheme="http",
    request_timeout=(10, 600),
    max_attempts=8
)

cursor = conn.cursor()

print("Подключение к Trino восстановлено!")


Подключение к Trino восстановлено!


In [72]:

stage_table = "lakehouse.stage.stg_tower_first_death"

# Проверяем, не была ли таблица создана предыдущим запуском
cursor.execute("SHOW TABLES FROM lakehouse.stage")
existing_tables = {row[0] for row in cursor.fetchall()}

if "stg_tower_first_death" in existing_tables:
    cursor.execute(f"SELECT COUNT(*) FROM {stage_table}")
    existing_rows = cursor.fetchone()[0]

    if existing_rows > 0:
        print(f"STAGE уже существует. Строк: {existing_rows}")
        skip_creation = True
    else:
        cursor.execute(f"DROP TABLE {stage_table}")
        skip_creation = False
else:
    skip_creation = False


if not skip_creation:

    # Новый уникальный путь в MinIO
    stage_location = (
        f"s3://warehouse/stage/stg_tower_first_death_{uuid.uuid4().hex}"
    )

    create_stage_query = f"""
    CREATE TABLE {stage_table}
    WITH (
        format = 'PARQUET',
        location = '{stage_location}'
    )
    AS

    WITH matches AS (
        SELECT
            match_id,
            duration,
            game_mode,
            lobby_type,
            radiant_win,

            CASE
                WHEN radiant_win THEN 'RADIANT'
                ELSE 'DIRE'
            END AS winner,

            duration < 900 AS is_short_match

        FROM lakehouse.raw.pg__matches

        WHERE match_id IS NOT NULL
          AND duration > 0
          AND radiant_win IS NOT NULL
    ),

    -- Сразу находим события, в которых присутствует T1
    t1_candidates AS (
        SELECT
            d.match_id,
            d.time,

            bitwise_and(
                CAST(d.dire AS BIGINT),
                BIGINT '7'
            ) AS dire_t1_mask,

            bitwise_and(
                CAST(d.radiant AS BIGINT),
                BIGINT '7'
            ) AS radiant_t1_mask

        FROM lakehouse.raw.mongo__tower_death_events d

        INNER JOIN matches m
            ON d.match_id = m.match_id

        WHERE d.id IS NOT NULL
          AND d.time IS NOT NULL
          AND d.time >= 0
          AND d.time <= m.duration
          AND d.dire IS NOT NULL
          AND d.radiant IS NOT NULL

          AND (
              bitwise_and(CAST(d.dire AS BIGINT), BIGINT '7') <> 0
              OR
              bitwise_and(CAST(d.radiant AS BIGINT), BIGINT '7') <> 0
          )
    ),

    -- Самое раннее время падения T1
    first_t1_times AS (
        SELECT
            match_id,
            MIN(time) AS first_t1_time

        FROM t1_candidates

        GROUP BY match_id
    ),

    -- Определяем уничтоженную сторону
    first_t1 AS (
        SELECT
            c.match_id,
            f.first_t1_time,

            MAX(
                CASE WHEN c.dire_t1_mask <> 0
                     THEN 1 ELSE 0 END
            ) AS dire_changed,

            MAX(
                CASE WHEN c.radiant_t1_mask <> 0
                     THEN 1 ELSE 0 END
            ) AS radiant_changed

        FROM t1_candidates c

        INNER JOIN first_t1_times f
            ON c.match_id = f.match_id
           AND c.time = f.first_t1_time

        GROUP BY
            c.match_id,
            f.first_t1_time
    )

    SELECT
        m.match_id,
        m.duration,
        m.game_mode,
        m.lobby_type,
        m.is_short_match,

        f.first_t1_time,

        CASE
            WHEN f.dire_changed = 1
             AND f.radiant_changed = 1 THEN 'BOTH'

            WHEN f.dire_changed = 1 THEN 'DIRE'

            WHEN f.radiant_changed = 1 THEN 'RADIANT'
        END AS destroyed_side,

        CASE
            WHEN f.dire_changed = 1
             AND f.radiant_changed = 0 THEN 'RADIANT'

            WHEN f.radiant_changed = 1
             AND f.dire_changed = 0 THEN 'DIRE'

            ELSE NULL
        END AS destroyer_team,

        m.radiant_win,
        m.winner,

        TRUE AS has_t1_event

    FROM matches m

    INNER JOIN first_t1 f
        ON m.match_id = f.match_id
    """

    cursor.execute(create_stage_query)

    print("STAGE создана!")
    print("MinIO:", stage_location)


# Проверяем результат
cursor.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT match_id) AS unique_matches,
        COUNT_IF(first_t1_time IS NULL) AS null_t1,
        COUNT_IF(destroyed_side = 'BOTH') AS simultaneous_events
    FROM {stage_table}
""")

columns = [c[0] for c in cursor.description]
display(pd.DataFrame(cursor.fetchall(), columns=columns))


STAGE уже существует. Строк: 314186


,total_rows,unique_matches,null_t1,simultaneous_events
0,314186,314186,0,532


In [73]:

# Загружаем первые 100 строк STAGE
cursor.execute("""
    SELECT *
    FROM lakehouse.stage.stg_tower_first_death
    ORDER BY first_t1_time
    LIMIT 100
""")

columns = [col[0] for col in cursor.description]
rows = cursor.fetchall()

df_stage = pd.DataFrame(rows, columns=columns)

display(df_stage)


,match_id,duration,game_mode,lobby_type,is_short_match,first_t1_time,destroyed_side,destroyer_team,radiant_win,winner,has_t1_event
0,9044806670,1581,22,0,False,47,DIRE,RADIANT,False,DIRE,True
1,9045432420,2465,23,0,False,56,DIRE,RADIANT,True,RADIANT,True
2,9044710427,1233,22,7,False,61,DIRE,RADIANT,True,RADIANT,True
3,9044864375,1272,22,7,False,62,DIRE,RADIANT,True,RADIANT,True
4,9045427669,945,22,7,False,62,RADIANT,DIRE,False,DIRE,True
...,...,...,...,...,...,...,...,...,...,...,...
95,9045249422,1547,23,0,False,122,RADIANT,DIRE,True,RADIANT,True
96,9045435477,1214,22,7,False,122,DIRE,RADIANT,True,RADIANT,True
97,9045327423,1135,23,0,False,123,RADIANT,DIRE,True,RADIANT,True
98,9045026845,1271,23,0,False,123,DIRE,RADIANT,True,RADIANT,True


In [74]:

from uuid import uuid4

mart_table = "lakehouse.mart.mart_t1_winrate_by_time"

# Новый уникальный путь для Iceberg в MinIO
mart_location = (
    f"s3://warehouse/mart/mart_t1_winrate_by_time_{uuid4().hex}"
)

# Удаляем старую таблицу MART, если она существует
cursor.execute(f"DROP TABLE IF EXISTS {mart_table}")

create_mart_query = f"""
CREATE TABLE {mart_table}
WITH (
    format = 'PARQUET',
    location = '{mart_location}'
)
AS

WITH bucketed AS (
    SELECT
        match_id,
        first_t1_time,
        destroyer_team,
        winner,
        game_mode,
        lobby_type,
        is_short_match,

        CASE
            WHEN first_t1_time < 300 THEN '00-05 минут'
            WHEN first_t1_time < 600 THEN '05-10 минут'
            WHEN first_t1_time < 900 THEN '10-15 минут'
            WHEN first_t1_time < 1200 THEN '15-20 минут'
            WHEN first_t1_time < 1500 THEN '20-25 минут'
            WHEN first_t1_time < 1800 THEN '25-30 минут'
            ELSE '30+ минут'
        END AS time_bucket,

        CASE
            WHEN first_t1_time < 300 THEN 1
            WHEN first_t1_time < 600 THEN 2
            WHEN first_t1_time < 900 THEN 3
            WHEN first_t1_time < 1200 THEN 4
            WHEN first_t1_time < 1500 THEN 5
            WHEN first_t1_time < 1800 THEN 6
            ELSE 7
        END AS bucket_order,

        CASE
            WHEN destroyer_team = winner THEN 1
            ELSE 0
        END AS is_win

    FROM lakehouse.stage.stg_tower_first_death

    WHERE destroyer_team IN ('RADIANT', 'DIRE')
      AND first_t1_time IS NOT NULL
)

SELECT
    time_bucket,
    bucket_order,
    destroyer_team,
    game_mode,
    lobby_type,
    is_short_match,

    COUNT(*) AS matches_count,

    SUM(is_win) AS wins_count,

    ROUND(
        100.0 * SUM(is_win) / COUNT(*),
        2
    ) AS win_rate_pct,

    SUM(first_t1_time) AS sum_first_t1_time_sec,

    ROUND(
        AVG(CAST(first_t1_time AS DOUBLE)),
        2
    ) AS avg_first_t1_time_sec

FROM bucketed

GROUP BY
    time_bucket,
    bucket_order,
    destroyer_team,
    game_mode,
    lobby_type,
    is_short_match
"""

cursor.execute(create_mart_query)

print("MART успешно пересоздана!")
print("Таблица:", mart_table)
print("MinIO:", mart_location)


MART успешно пересоздана!
Таблица: lakehouse.mart.mart_t1_winrate_by_time
MinIO: s3://warehouse/mart/mart_t1_winrate_by_time_4be4f90e28f74c94b02974e008dfa409


In [75]:

query = """
SELECT
    time_bucket,
    bucket_order,

    SUM(matches_count) AS matches_count,
    SUM(wins_count) AS wins_count,

    ROUND(
        100.0 * SUM(wins_count) / SUM(matches_count),
        2
    ) AS win_rate_pct,

    ROUND(
        CAST(SUM(sum_first_t1_time_sec) AS DOUBLE)
        / SUM(matches_count),
        2
    ) AS avg_first_t1_time_sec

FROM lakehouse.mart.mart_t1_winrate_by_time

GROUP BY
    time_bucket,
    bucket_order

ORDER BY bucket_order
"""

cursor.execute(query)

columns = [c[0] for c in cursor.description]
df_mart_overall = pd.DataFrame(
    cursor.fetchall(),
    columns=columns
)

display(df_mart_overall)


,time_bucket,bucket_order,matches_count,wins_count,win_rate_pct,avg_first_t1_time_sec
0,00-05 минут,1,10505,6881,65.5,254.71
1,05-10 минут,2,158090,98258,62.2,453.22
2,10-15 минут,3,128892,80593,62.5,712.81
3,15-20 минут,4,15811,9580,60.6,973.38
4,20-25 минут,5,337,197,58.5,1277.81
5,25-30 минут,6,14,10,71.4,1608.86
6,30+ минут,7,5,3,60.0,2062.00


In [76]:

# ==========================================
# QUALITY CHECKS: STAGE + MART + ICEBERG
# ==========================================

stage_table = "lakehouse.stage.stg_tower_first_death"
mart_table = "lakehouse.mart.mart_t1_winrate_by_time"


def run_quality_check(name, query):
    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)

    cursor.execute(query)

    columns = [c[0] for c in cursor.description]
    result = pd.DataFrame(cursor.fetchall(), columns=columns)

    display(result)

    return result


# 1. Проверка STAGE
stage_quality = run_quality_check(
    "1. КАЧЕСТВО STAGE",
    f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT match_id) AS unique_matches,

        COUNT(*) - COUNT(DISTINCT match_id)
            AS duplicate_matches,

        COUNT_IF(match_id IS NULL)
            AS null_match_ids,

        COUNT_IF(first_t1_time IS NULL OR first_t1_time < 0)
            AS invalid_t1_time,

        COUNT_IF(first_t1_time > duration)
            AS t1_after_match_end,

        COUNT_IF(is_short_match <> (duration < 900))
            AS incorrect_short_match_flag,

        COUNT_IF(
            destroyed_side IS NULL
            OR destroyed_side NOT IN ('DIRE', 'RADIANT', 'BOTH')
        ) AS invalid_destroyed_side,

        COUNT_IF(
            destroyed_side = 'DIRE'
            AND (destroyer_team IS NULL OR destroyer_team <> 'RADIANT')
        ) AS incorrect_dire_mapping,

        COUNT_IF(
            destroyed_side = 'RADIANT'
            AND (destroyer_team IS NULL OR destroyer_team <> 'DIRE')
        ) AS incorrect_radiant_mapping

    FROM {stage_table}
    """
)


# 2. Проверка MART
mart_quality = run_quality_check(
    "2. КАЧЕСТВО MART",
    f"""
    SELECT
        COUNT(*) AS total_groups,

        COUNT_IF(
            win_rate_pct IS NULL
            OR win_rate_pct < 0
            OR win_rate_pct > 100
        ) AS invalid_win_rates,

        COUNT_IF(matches_count <= 0)
            AS invalid_match_counts,

        COUNT_IF(
            wins_count < 0
            OR wins_count > matches_count
        ) AS invalid_wins,

        COUNT_IF(
            ABS(
                win_rate_pct -
                ROUND(100.0 * wins_count / matches_count, 2)
            ) > 0.011
        ) AS incorrect_win_rate_calculations

    FROM {mart_table}
    """
)


# 3. Сверка количества матчей между STAGE и MART
cross_layer = run_quality_check(
    "3. СВЕРКА STAGE И MART",
    f"""
    SELECT

        (SELECT COUNT(*)
         FROM {stage_table}) AS stage_rows,

        (SELECT COUNT(*)
         FROM {stage_table}
         WHERE destroyer_team IN ('DIRE', 'RADIANT')
        ) AS eligible_stage_matches,

        (SELECT SUM(matches_count)
         FROM {mart_table}) AS mart_matches,

        (SELECT COUNT(*)
         FROM {stage_table}
         WHERE destroyed_side = 'BOTH'
        ) AS simultaneous_events
    """
)


# 4. Проверяем историю Iceberg STAGE
print("\n" + "=" * 75)
print("4. ICEBERG SNAPSHOTS — STAGE")
print("=" * 75)

try:
    cursor.execute("""
        SELECT
            snapshot_id,
            committed_at,
            operation
        FROM lakehouse.stage."stg_tower_first_death$snapshots"
        ORDER BY committed_at DESC
        LIMIT 5
    """)

    columns = [c[0] for c in cursor.description]
    display(pd.DataFrame(cursor.fetchall(), columns=columns))

except Exception as error:
    print("Не удалось прочитать snapshots STAGE:", error)


# 5. Проверяем историю Iceberg MART
print("\n" + "=" * 75)
print("5. ICEBERG SNAPSHOTS — MART")
print("=" * 75)

try:
    cursor.execute("""
        SELECT
            snapshot_id,
            committed_at,
            operation
        FROM lakehouse.mart."mart_t1_winrate_by_time$snapshots"
        ORDER BY committed_at DESC
        LIMIT 5
    """)

    columns = [c[0] for c in cursor.description]
    display(pd.DataFrame(cursor.fetchall(), columns=columns))

except Exception as error:
    print("Не удалось прочитать snapshots MART:", error)


# 6. Показываем физическую конфигурацию Iceberg
print("\n" + "=" * 75)
print("6. SHOW CREATE TABLE — MART")
print("=" * 75)

cursor.execute(f"SHOW CREATE TABLE {mart_table}")

for row in cursor.fetchall():
    print(row[0])



1. КАЧЕСТВО STAGE


,total_rows,unique_matches,duplicate_matches,null_match_ids,invalid_t1_time,t1_after_match_end,incorrect_short_match_flag,invalid_destroyed_side,incorrect_dire_mapping,incorrect_radiant_mapping
0,314186,314186,0,0,0,0,0,0,0,0



2. КАЧЕСТВО MART


,total_groups,invalid_win_rates,invalid_match_counts,invalid_wins,incorrect_win_rate_calculations
0,110,0,0,0,0



3. СВЕРКА STAGE И MART


,stage_rows,eligible_stage_matches,mart_matches,simultaneous_events
0,314186,313654,313654,532



4. ICEBERG SNAPSHOTS — STAGE


,snapshot_id,committed_at,operation
0,5698965369267968477,2026-11-13 12:41:26.703000+07:00,append



5. ICEBERG SNAPSHOTS — MART


,snapshot_id,committed_at,operation
0,2624722839376831498,2026-11-13 12:55:22.860000+07:00,append



6. SHOW CREATE TABLE — MART
CREATE TABLE lakehouse.mart.mart_t1_winrate_by_time (
   time_bucket varchar,
   bucket_order integer,
   destroyer_team varchar,
   game_mode integer,
   lobby_type integer,
   is_short_match boolean,
   matches_count bigint,
   wins_count bigint,
   win_rate_pct decimal(24, 1),
   sum_first_t1_time_sec bigint,
   avg_first_t1_time_sec double
)
WITH (
   format = 'PARQUET',
   format_version = 2,
   location = 's3://warehouse/mart/mart_t1_winrate_by_time_4be4f90e28f74c94b02974e008dfa409',
   max_commit_retry = 4
)
